In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/multilingual-speech-recognition/sample_submission.csv
/kaggle/input/competitions/multilingual-speech-recognition/train.csv
/kaggle/input/competitions/multilingual-speech-recognition/test.csv
/kaggle/input/competitions/multilingual-speech-recognition/competition_data/test/audio_00000.wav
/kaggle/input/competitions/multilingual-speech-recognition/competition_data/test/audio_00019.wav
/kaggle/input/competitions/multilingual-speech-recognition/competition_data/test/audio_00088.wav
/kaggle/input/competitions/multilingual-speech-recognition/competition_data/test/audio_00071.wav
/kaggle/input/competitions/multilingual-speech-recognition/competition_data/test/audio_00084.wav
/kaggle/input/competitions/multilingual-speech-recognition/competition_data/test/audio_00001.wav
/kaggle/input/competitions/multilingual-speech-recognition/competition_data/test/audio_00031.wav
/kaggle/input/competitions/multilingual-speech-recognition/competition_data/test/audio_00094.wav
/kaggl

In [2]:
!pip install -q transformers datasets librosa jiwer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 39.1 MB/s eta 0:00:00


In [3]:
train_df = pd.read_csv('/kaggle/input/competitions/multilingual-speech-recognition/train.csv')
test_df = pd.read_csv('/kaggle/input/competitions/multilingual-speech-recognition/test.csv')

train_df.head()

,id,audio,text
0,0,audio_00000.wav,you had quoted plutarch line.
1,1,audio_00001.wav,மலையேறுதலில் வந்து பார்த்தீங்கன்னா ஜஸ்ட்டு நம்...
2,2,audio_00002.wav,to do his phd in engineering about four years ...
3,3,audio_00003.wav,maybe he was not at home.
4,4,audio_00004.wav,BUT WE DIDN'T BREAK HIS OLD WINDOW YOU KNOW EX...


In [4]:
import librosa

BASE_PATH = "/kaggle/input/competitions/multilingual-speech-recognition/competition_data/train/"

def load_audio(filename):
    full_path = BASE_PATH + filename
    audio, sr = librosa.load(full_path, sr=16000)
    audio = librosa.util.normalize(audio)
    return audio

In [5]:
import re

def normalize_text(text):
    text = text.lower()
    text = re.sub(r"[^\w\s]", "", text)  # remove punctuation
    text = re.sub(r"\s+", " ", text).strip()
    return text

train_df['text'] = train_df['text'].apply(normalize_text)

In [6]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration

model_name = "openai/whisper-small"

processor = WhisperProcessor.from_pretrained(model_name)
model = WhisperForConditionalGeneration.from_pretrained(model_name)

preprocessor_config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

In [7]:
from datasets import Dataset

def prepare(example):
    audio = load_audio(example['audio'])
    
    inputs = processor(audio, sampling_rate=16000, return_tensors="pt")
    example["input_features"] = inputs.input_features[0]
    
    labels = processor.tokenizer(example["text"]).input_ids
    example["labels"] = labels
    
    return example

dataset = Dataset.from_pandas(train_df)
dataset = dataset.map(prepare)

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [8]:
from transformers import DataCollatorForSeq2Seq

def data_collator(features):
    input_features = [torch.tensor(f["input_features"]) for f in features]
    labels = [torch.tensor(f["labels"]) for f in features]

    input_features = torch.nn.utils.rnn.pad_sequence(
        input_features, batch_first=True
    )

    labels = torch.nn.utils.rnn.pad_sequence(
        labels,
        batch_first=True,
        padding_value=-100
    )

    return {
        "input_features": input_features,
        "labels": labels
    }

In [9]:
from transformers import Seq2SeqTrainingArguments,Seq2SeqTrainer
import torch

training_args = Seq2SeqTrainingArguments(
    output_dir="./whisper",
    per_device_train_batch_size=8,
    learning_rate=1e-5,
    num_train_epochs=2,
    fp16=True,
    logging_steps=100,
    save_steps=500
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    data_collator=data_collator,
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss
100,2.371475
200,1.272388


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=250, training_loss=1.6756167755126954, metrics={'train_runtime': 1637.6947, 'train_samples_per_second': 2.442, 'train_steps_per_second': 0.153, 'total_flos': 1.15434160128e+18, 'train_loss': 1.6756167755126954, 'epoch': 2.0})

In [10]:
from jiwer import wer

def predict(audio_path):
    audio = load_audio(audio_path)

    inputs = processor(audio, sampling_rate=16000, return_tensors="pt")
    input_features = inputs.input_features.to(model.device)

    predicted_ids = model.generate(
        input_features,
        language="en",
        task="transcribe"
    )

    transcription = processor.batch_decode(
        predicted_ids, skip_special_tokens=True
    )[0]

    return transcription

In [11]:
preds = []
refs = []

for _, row in train_df.sample(100).iterrows():
    pred = predict(row['audio'])
    preds.append(pred)
    refs.append(row['text'])

print("WER:", wer(refs, preds))

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensA

WER: 0.13425925925925927


In [12]:
test_df['text'] = test_df['audio'].apply(predict)

test_df['text'] = test_df['text'].apply(normalize_text)

test_df['text'] = test_df['text'].apply(lambda x: x if x.strip() != "" else "unknown")

submission = test_df[['audio', 'text']]

submission.to_csv("submission.csv", index=False)